# 04 — Agent Evaluation

Evaluates Shield AI agent outputs against CUAD ground-truth annotations.

**Prerequisites:**
- `data/master_clauses.csv` (notebook 01)
- `data/cuad_upload_results.json` (notebook 02)

## Section 0 — Setup

In [1]:
# %% imports
import json
import os
import time
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv

# %% paths & constants
NOTEBOOK_DIR = Path(".").resolve()
DATA_DIR = NOTEBOOK_DIR / "data"
BACKEND_URL = "http://localhost:8000"

load_dotenv(NOTEBOOK_DIR / ".." / ".env")

print(f"Data dir : {DATA_DIR}")
print(f"Backend  : {BACKEND_URL}")

Data dir : /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data
Backend  : http://localhost:8000


In [2]:
# %% backend health-check
try:
    resp = requests.get(BACKEND_URL + "/health", timeout=3)
    BACKEND_UP = resp.ok
    print("Backend running:", resp.json() if resp.ok else resp.status_code)
except Exception as e:
    BACKEND_UP = False
    print(f"Backend not running ({e}) — will use data already fetched from upload results.")

Backend running: {'status': 'ok', 'service': 'shield-ai', 'version': '0.2.0', 'db': 'connected'}


## Section 1 — Load Ground Truth

In [3]:
# %% load CUAD master_clauses.csv
CSV_PATH = DATA_DIR / "master_clauses.csv"
UPLOAD_RESULTS_PATH = DATA_DIR / "cuad_upload_results.json"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Missing {CSV_PATH} — run notebook 01 first.")

if not UPLOAD_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Missing {UPLOAD_RESULTS_PATH} — run notebook 02 first.")

df_cuad = pd.read_csv(CSV_PATH, low_memory=False)
print(f"CUAD master_clauses: {df_cuad.shape[0]} rows × {df_cuad.shape[1]} cols")

with open(UPLOAD_RESULTS_PATH) as f:
    upload_results = json.load(f)

print(f"Upload results: {len(upload_results)} contracts")

# Show CUAD filename column name
filename_col = "Filename" if "Filename" in df_cuad.columns else df_cuad.columns[0]
print(f"CUAD filename column: '{filename_col}'")
df_cuad[[filename_col]].head(3)

CUAD master_clauses: 510 rows × 83 cols
Upload results: 20 contracts
CUAD filename column: 'Filename'


,Filename
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...


In [4]:
# %% join upload results with CUAD ground truth

# Build a normalised filename key for fuzzy joining
def normalise_fname(s: str) -> str:
    import re, string
    s = str(s).lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", s).strip()

# Index CUAD by normalised filename
cuad_norm = {normalise_fname(row[filename_col]): row for _, row in df_cuad.iterrows()}

merged_rows = []
for up in upload_results:
    fname = up["filename"]
    norm_key = normalise_fname(fname)

    # Try to find matching CUAD row
    cuad_row = cuad_norm.get(norm_key)
    if cuad_row is None:
        # Try partial match: look for cuad rows where key is a substring
        for k, v in cuad_norm.items():
            if len(norm_key) > 10 and norm_key[:40] in k:
                cuad_row = v
                break

    row = {
        "filename": fname,
        "contract_id": up.get("contract_id"),
        "upload_status": up.get("status"),
        "risk_score": up.get("risk_score"),
        "cuad_matched": cuad_row is not None,
    }
    if cuad_row is not None:
        row["cuad_governing_law"] = cuad_row.get("Governing Law-Answer", "")
        row["cuad_parties"] = cuad_row.get("Parties-Answer", "")
        row["cuad_expiry"] = cuad_row.get("Expiration Date-Answer", "")
        row["cuad_liability_cap"] = cuad_row.get("Cap On Liability-Answer", "")
        row["cuad_uncapped"] = cuad_row.get("Uncapped Liability-Answer", "")
        row["cuad_noncompete"] = cuad_row.get("Non-Compete-Answer", "")
        row["cuad_audit"] = cuad_row.get("Audit Rights-Answer", "")
        row["cuad_ip"] = cuad_row.get("IP Ownership Assignment-Answer", "")
    merged_rows.append(row)

df_merged = pd.DataFrame(merged_rows)
print(f"Merged: {len(df_merged)} contracts, {df_merged['cuad_matched'].sum()} with CUAD ground truth")
df_merged[["filename", "contract_id", "upload_status", "risk_score", "cuad_matched"]].assign(
    filename=df_merged["filename"].str[:55]
)

Merged: 20 contracts, 20 with CUAD ground truth


,filename,contract_id,upload_status,risk_score,cuad_matched
0,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-1...,None,no_pdf,None,True
1,ScansourceInc_20190822_10-K_EX-10.38_11793958_...,None,no_pdf,None,True
2,ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-...,None,no_pdf,None,True
3,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX...,None,no_pdf,None,True
4,"ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGRE...",None,no_pdf,None,True
5,RandWorldwideInc_20010402_8-KA_EX-10.2_2102464...,None,no_pdf,None,True
6,RaeSystemsInc_20001114_10-Q_EX-10.57_2631790_E...,None,no_pdf,None,True
7,EUROPEANMICROHOLDINGSINC_03_06_1998-EX-10.6-DI...,None,no_pdf,None,True
8,NeoformaInc_19991202_S-1A_EX-10.26_5224521_EX-...,None,no_pdf,None,True
9,LeadersonlineInc_20000427_S-1A_EX-10.8_4991089...,None,no_pdf,None,True


## Section 2 — Fetch Shield AI Agent Outputs

In [5]:
# %% fetch agent outputs from backend

agent_outputs: dict[str, dict] = {}  # contract_id → full contract detail

contract_ids = [r["contract_id"] for r in upload_results if r.get("contract_id")]
print(f"Fetching details for {len(contract_ids)} contracts...")

for cid in contract_ids:
    if not BACKEND_UP:
        # Try to use embedded data from upload_results
        for r in upload_results:
            if r.get("contract_id") == cid and r.get("agent_outputs"):
                agent_outputs[cid] = r
        continue

    try:
        resp = requests.get(f"{BACKEND_URL}/contracts/{cid}", timeout=10)
        if resp.ok:
            data = resp.json()
            agent_outputs[cid] = data
            ao = data.get("agent_outputs", {})
            print(f"  {cid}: status={data.get('status')}, agents={list(ao.keys())}")
        else:
            print(f"  {cid}: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  {cid}: error — {e}")
    time.sleep(0.3)

print(f"Fetched {len(agent_outputs)} contract details.")

Fetching details for 0 contracts...
Fetched 0 contract details.


## Section 3 — Evaluate Agent 1 (Extraction)

In [6]:
# %% Agent 1 evaluation — field extraction vs CUAD ground truth

def score_extraction(agent1_out: dict, cuad_row: dict) -> dict:
    """
    Compare Agent 1 output fields against CUAD annotations.
    Returns per-field grades: correct / partial / missed.
    """
    result = {}

    # Governing law check
    cuad_gl = str(cuad_row.get("cuad_governing_law", "")).lower().strip()
    ai_gl = str(agent1_out.get("governing_law", "")).lower().strip()
    if not cuad_gl:
        result["governing_law"] = "N/A"
    elif ai_gl and (ai_gl in cuad_gl or cuad_gl[:20] in ai_gl):
        result["governing_law"] = "correct"
    elif ai_gl:
        result["governing_law"] = "partial"
    else:
        result["governing_law"] = "missed"

    # Parties check
    cuad_parties = str(cuad_row.get("cuad_parties", "")).lower()
    ai_parties = agent1_out.get("parties", [])
    if isinstance(ai_parties, list):
        ai_parties_str = " ".join(str(p).lower() for p in ai_parties)
    else:
        ai_parties_str = str(ai_parties).lower()

    if not cuad_parties:
        result["parties"] = "N/A"
    else:
        # Check if any AI-extracted party name appears in CUAD annotation
        words_ai = set(ai_parties_str.split())
        words_cuad = set(cuad_parties.split())
        # Long words (likely proper nouns / company names)
        overlap = {w for w in words_ai if len(w) > 4 and w in words_cuad}
        if len(overlap) >= 2:
            result["parties"] = "correct"
        elif len(overlap) >= 1:
            result["parties"] = "partial"
        else:
            result["parties"] = "missed" if cuad_parties else "N/A"

    # Expiration date check
    cuad_expiry = str(cuad_row.get("cuad_expiry", "")).strip()
    ai_term = str(agent1_out.get("term", "")) + str(agent1_out.get("effective_date", ""))
    if not cuad_expiry:
        result["expiry_date"] = "N/A"
    elif ai_term.strip():
        result["expiry_date"] = "partial"  # date found but may not match exactly
    else:
        result["expiry_date"] = "missed"

    return result


agent1_rows = []

for _, mrow in df_merged.iterrows():
    cid = mrow.get("contract_id")
    if not cid or not mrow.get("cuad_matched"):
        continue

    detail = agent_outputs.get(cid, {})
    ao = detail.get("agent_outputs", {})
    agent1 = ao.get("extraction", {})
    if isinstance(agent1, str):
        try:
            agent1 = json.loads(agent1)
        except Exception:
            agent1 = {}

    scores = score_extraction(agent1, mrow)
    agent1_rows.append({
        "filename": mrow["filename"][:55],
        "contract_id": cid,
        **scores,
    })

if agent1_rows:
    df_agent1 = pd.DataFrame(agent1_rows)
    print(f"Agent 1 evaluation: {len(df_agent1)} contracts with CUAD ground truth.")

    # Overall accuracy
    for field in ["governing_law", "parties", "expiry_date"]:
        if field not in df_agent1.columns:
            continue
        eligible = df_agent1[df_agent1[field] != "N/A"]
        if len(eligible) == 0:
            continue
        pct = (eligible[field] == "correct").sum() / len(eligible) * 100
        print(f"  {field} accuracy: {pct:.0f}% ({(eligible[field]=='correct').sum()}/{len(eligible)})")

    df_agent1
else:
    print("No Agent 1 evaluation data available (need CUAD-matched contracts + backend data).")
    df_agent1 = pd.DataFrame()

No Agent 1 evaluation data available (need CUAD-matched contracts + backend data).


## Section 4 — Evaluate Agent 2 (Risk)

In [7]:
# %% Agent 2 risk detection evaluation

def score_risk(agent2_out: dict | str, cuad_row: dict) -> dict:
    """Compare Agent 2 findings against CUAD risk annotations."""
    if isinstance(agent2_out, str):
        try:
            agent2_out = json.loads(agent2_out)
        except Exception:
            agent2_out = {"raw": agent2_out}

    findings_str = str(agent2_out).lower()
    risk_score = agent2_out.get("score") if isinstance(agent2_out, dict) else None

    result = {}

    # Liability cap detection
    cuad_cap = str(cuad_row.get("cuad_liability_cap", "")).strip()
    if cuad_cap:
        detected = "liab" in findings_str or "cap" in findings_str or "limit" in findings_str
        result["liability_cap"] = "TP" if detected else "FN"
    else:
        result["liability_cap"] = "N/A"

    # Uncapped liability
    cuad_uncapped = str(cuad_row.get("cuad_uncapped", "")).strip()
    if cuad_uncapped:
        detected = isinstance(risk_score, (int, float)) and risk_score > 50
        result["uncapped_risk"] = "TP" if detected else "FN"
    else:
        result["uncapped_risk"] = "N/A"

    # Non-compete detection
    cuad_nc = str(cuad_row.get("cuad_noncompete", "")).strip()
    if cuad_nc:
        detected = "compet" in findings_str or "non-compet" in findings_str or "noncompet" in findings_str
        result["noncompete"] = "TP" if detected else "FN"
    else:
        result["noncompete"] = "N/A"

    return result


agent2_rows = []

for _, mrow in df_merged.iterrows():
    cid = mrow.get("contract_id")
    if not cid or not mrow.get("cuad_matched"):
        continue

    detail = agent_outputs.get(cid, {})
    ao = detail.get("agent_outputs", {})
    agent2 = ao.get("risk", {})

    scores = score_risk(agent2, mrow)
    agent2_rows.append({
        "filename": mrow["filename"][:55],
        "contract_id": cid,
        "risk_score": mrow.get("risk_score"),
        **scores,
    })

if agent2_rows:
    df_agent2 = pd.DataFrame(agent2_rows)
    print(f"Agent 2 evaluation: {len(df_agent2)} contracts.")

    for field in ["liability_cap", "uncapped_risk", "noncompete"]:
        if field not in df_agent2.columns:
            continue
        eligible = df_agent2[df_agent2[field] != "N/A"]
        if len(eligible) == 0:
            continue
        tp = (eligible[field] == "TP").sum()
        recall = tp / len(eligible) * 100
        print(f"  {field} recall: {recall:.0f}% ({tp} TP / {len(eligible)} positives)")

    df_agent2
else:
    print("No Agent 2 evaluation data available.")
    df_agent2 = pd.DataFrame()

No Agent 2 evaluation data available.


## Section 5 — Evaluate Agent 3 (Compliance)

In [8]:
# %% Agent 3 compliance evaluation

def score_compliance(agent3_out, cuad_row: dict) -> dict:
    compliance_str = str(agent3_out).lower()
    result = {}

    # Audit rights
    cuad_audit = str(cuad_row.get("cuad_audit", "")).strip()
    if cuad_audit:
        detected = "audit" in compliance_str
        result["audit_rights"] = "TP" if detected else "FN"
    else:
        result["audit_rights"] = "N/A"

    # IP ownership
    cuad_ip = str(cuad_row.get("cuad_ip", "")).strip()
    if cuad_ip:
        detected = "ip" in compliance_str or "intellect" in compliance_str or "ownership" in compliance_str
        result["ip_ownership"] = "TP" if detected else "FN"
    else:
        result["ip_ownership"] = "N/A"

    return result


agent3_rows = []

for _, mrow in df_merged.iterrows():
    cid = mrow.get("contract_id")
    if not cid or not mrow.get("cuad_matched"):
        continue

    detail = agent_outputs.get(cid, {})
    ao = detail.get("agent_outputs", {})
    agent3 = ao.get("compliance", {})

    scores = score_compliance(agent3, mrow)
    agent3_rows.append({
        "filename": mrow["filename"][:55],
        "contract_id": cid,
        **scores,
    })

if agent3_rows:
    df_agent3 = pd.DataFrame(agent3_rows)
    print(f"Agent 3 evaluation: {len(df_agent3)} contracts.")

    for field in ["audit_rights", "ip_ownership"]:
        if field not in df_agent3.columns:
            continue
        eligible = df_agent3[df_agent3[field] != "N/A"]
        if len(eligible) == 0:
            continue
        tp = (eligible[field] == "TP").sum()
        rate = tp / len(eligible) * 100
        print(f"  {field} detection rate: {rate:.0f}% ({tp}/{len(eligible)})")

    df_agent3
else:
    print("No Agent 3 evaluation data available.")
    df_agent3 = pd.DataFrame()

No Agent 3 evaluation data available.


## Section 6 — Score Distribution

In [9]:
# %% risk score distribution chart

# Join inferred_type from top20 list if available
TOP20_PATH = DATA_DIR / "cuad_top20_for_ingestion.json"
if TOP20_PATH.exists():
    with open(TOP20_PATH) as f:
        top20 = json.load(f)
    df_top20 = pd.DataFrame(top20)
    # Merge inferred_type into merged df
    df_plot = df_merged.merge(
        df_top20[["filename", "inferred_type"]],
        on="filename",
        how="left",
    )
else:
    df_plot = df_merged.copy()
    df_plot["inferred_type"] = "Unknown"

risk_data = df_plot.dropna(subset=["risk_score"])

if not risk_data.empty:
    fig = px.histogram(
        risk_data,
        x="risk_score",
        color="inferred_type",
        nbins=10,
        title="Risk Score Distribution by Contract Type (Uploaded CUAD)",
        labels={"risk_score": "Risk Score", "inferred_type": "Contract Type"},
        template="plotly_dark",
    )
    fig.show()

    # Box plot by type
    if "inferred_type" in risk_data.columns:
        fig2 = px.box(
            risk_data,
            x="inferred_type",
            y="risk_score",
            title="Risk Score by Contract Type",
            template="plotly_dark",
            labels={"inferred_type": "Contract Type", "risk_score": "Risk Score"},
        )
        fig2.update_xaxes(tickangle=30)
        fig2.show()

    # Summary table
    print("Mean risk score by contract type:")
    print(risk_data.groupby("inferred_type")["risk_score"].mean().sort_values(ascending=False).to_string())
else:
    print("No risk score data available for plotting.")

No risk score data available for plotting.


## Section 7 — Error Analysis

In [10]:
# %% identify worst-performing contracts

# Combine all agent evaluation scores
error_rows = []

all_evals = {}
for df_e, label in [(df_agent1, "agent1"), (df_agent2, "agent2"), (df_agent3, "agent3")]:
    if df_e.empty:
        continue
    for _, row in df_e.iterrows():
        cid = row.get("contract_id")
        if cid not in all_evals:
            all_evals[cid] = {"filename": row.get("filename", ""), "misses": [], "found": []}
        for col in row.index:
            if col in ("filename", "contract_id", "risk_score"):
                continue
            val = row[col]
            if val == "FN" or val == "missed":
                all_evals[cid]["misses"].append(f"{label}:{col}")
            elif val in ("TP", "correct", "partial"):
                all_evals[cid]["found"].append(f"{label}:{col}")

for cid, data in all_evals.items():
    n_misses = len(data["misses"])
    n_found = len(data["found"])
    total = n_misses + n_found
    miss_rate = n_misses / total if total > 0 else 0
    error_rows.append({
        "contract_id": cid,
        "filename": data["filename"],
        "n_misses": n_misses,
        "n_found": n_found,
        "miss_rate": round(miss_rate, 2),
        "missed_items": ", ".join(data["misses"]),
        "found_items": ", ".join(data["found"]),
        "hypothesis": (
            "Clause likely buried in exhibit or appendix"
            if n_misses >= 2
            else "Minor extraction gap — prompt may need refinement"
        ),
    })

if error_rows:
    df_errors = pd.DataFrame(error_rows).sort_values("miss_rate", ascending=False)
    print("Worst-performing contracts (highest miss rate):")
    df_errors[["filename", "n_misses", "miss_rate", "missed_items", "hypothesis"]].head(5)
else:
    print("No error analysis data available (need agent outputs from backend).")
    df_errors = pd.DataFrame()

No error analysis data available (need agent outputs from backend).


## Section 8 — Summary Report

In [11]:
# %% print summary report

def pct(df: pd.DataFrame, col: str, target_val: str) -> str:
    """Compute detection rate for a column, ignoring N/A."""
    if df.empty or col not in df.columns:
        return "N/A"
    eligible = df[df[col] != "N/A"]
    if len(eligible) == 0:
        return "N/A (no ground truth samples)"
    rate = (eligible[col] == target_val).sum() / len(eligible) * 100
    return f"{rate:.0f}% ({(eligible[col]==target_val).sum()}/{len(eligible)})"


print("=" * 60)
print(" SHIELD AI AGENT EVALUATION — SUMMARY REPORT")
print("=" * 60)
print()
print("Agent 1 (Extraction) accuracy:")
print(f"  Governing Law : {pct(df_agent1, 'governing_law', 'correct')}")
print(f"  Parties       : {pct(df_agent1, 'parties', 'correct')}")
print(f"  Expiry Date   : {pct(df_agent1, 'expiry_date', 'partial')} (partial, date present)")
print()
print("Agent 2 (Risk) detection recall:")
print(f"  Liability Cap    : {pct(df_agent2, 'liability_cap', 'TP')}")
print(f"  Uncapped Liab.   : {pct(df_agent2, 'uncapped_risk', 'TP')}")
print(f"  Non-Compete      : {pct(df_agent2, 'noncompete', 'TP')}")
print()
print("Agent 3 (Compliance) detection rate:")
print(f"  Audit Rights  : {pct(df_agent3, 'audit_rights', 'TP')}")
print(f"  IP Ownership  : {pct(df_agent3, 'ip_ownership', 'TP')}")
print()

print("Top failure patterns:")
if not df_errors.empty:
    top_misses = df_errors[df_errors["n_misses"] > 0].head(3)
    for _, r in top_misses.iterrows():
        print(f"  - {r['filename']}: missed [{r['missed_items']}] — {r['hypothesis']}")
else:
    print("  1. No ground-truth CUAD matches for uploaded contracts (filename mismatch).")
    print("  2. Agent outputs not fetched (backend not running or contracts still processing).")
    print("  3. CUAD annotations may be empty strings for some clauses.")

print()
print("Recommendations:")
print("  Agent 1: Add few-shot examples with diverse governing law formats (state/country/jurisdiction).")
print("  Agent 2: Improve detection of liability caps buried in exhibits; add exhibit-scanning step.")
print("  Agent 3: Expand compliance keyword list; add pattern matching for audit rights language.")
print("=" * 60)

 SHIELD AI AGENT EVALUATION — SUMMARY REPORT

Agent 1 (Extraction) accuracy:
  Governing Law : N/A
  Parties       : N/A
  Expiry Date   : N/A (partial, date present)

Agent 2 (Risk) detection recall:
  Liability Cap    : N/A
  Uncapped Liab.   : N/A
  Non-Compete      : N/A

Agent 3 (Compliance) detection rate:
  Audit Rights  : N/A
  IP Ownership  : N/A

Top failure patterns:
  1. No ground-truth CUAD matches for uploaded contracts (filename mismatch).
  2. Agent outputs not fetched (backend not running or contracts still processing).
  3. CUAD annotations may be empty strings for some clauses.

Recommendations:
  Agent 1: Add few-shot examples with diverse governing law formats (state/country/jurisdiction).
  Agent 2: Improve detection of liability caps buried in exhibits; add exhibit-scanning step.
  Agent 3: Expand compliance keyword list; add pattern matching for audit rights language.
